# ModelArts 昇腾环境检查
确认 NPU 可用性、CANN/驱动版本、Python 依赖

In [1]:
import torch
import torch_npu
import os, sys, platform

/home/ma-user/anaconda3/envs/PyTorch-2.7.1/lib/python3.12/site-packages/torch_npu/utils/collect_env.py:58: UserWarning: Warning: The /usr/local/Ascend/cann-8.5.2 owner does not match the current owner.
  warnings.warn(f"Warning: The {path} owner does not match the current owner.")
/home/ma-user/anaconda3/envs/PyTorch-2.7.1/lib/python3.12/site-packages/torch_npu/utils/collect_env.py:58: UserWarning: Warning: The /usr/local/Ascend/cann-8.5.2/aarch64-linux/ascend_toolkit_install.info owner does not match the current owner.
  warnings.warn(f"Warning: The {path} owner does not match the current owner.")
/home/ma-user/anaconda3/envs/PyTorch-2.7.1/lib/python3.12/site-packages/torch_npu/utils/collect_env.py:58: UserWarning: Warning: The /usr/local/Ascend/cann-8.5.2 owner does not match the current owner.
  warnings.warn(f"Warning: The {path} owner does not match the current owner.")
/home/ma-user/anaconda3/envs/PyTorch-2.7.1/lib/python3.12/site-packages/torch_npu/utils/collect_env.py:58: UserW

## 1. NPU 基本信息

In [2]:
print(f"NPU 可用:        {torch.npu.is_available()}")
print(f"NPU 数量:        {torch.npu.device_count()}")
print(f"NPU 名称:        {torch.npu.get_device_name(0)}")
print(f"NPU 显存总量:     {torch.npu.get_device_properties(0).total_memory / 1024**3:.1f} GB")

NPU 可用:        True
NPU 数量:        1
NPU 名称:        Ascend910B4
NPU 显存总量:     29.5 GB


## 2. PyTorch 与关键包

In [3]:
print(f"Python:       {sys.version.split()[0]}")
print(f"PyTorch:      {torch.__version__}")
print(f"torch_npu:    {torch_npu.__version__}")

try:
    import torchvision; print(f"torchvision:  {torchvision.__version__}")
except Exception as e: print(f"torchvision:  未安装 ({e})")

try:
    import numpy; print(f"numpy:        {numpy.__version__}")
except Exception as e: print(f"numpy:        未安装 ({e})")

try:
    import PIL; print(f"Pillow:       {PIL.__version__}")
except Exception as e: print(f"Pillow:       未安装 ({e})")

Python:       3.12.0
PyTorch:      2.7.1+cpu
torch_npu:    2.7.1.post2
torchvision:  0.22.1
numpy:        1.26.4
Pillow:       11.3.0


## 3. 系统信息

In [4]:
print(f"系统:          {platform.system()} {platform.release()}")
print(f"架构:          {platform.machine()}")
print(f"主机名:        {platform.node()}")
print(f"当前用户:      {os.environ.get('USER', 'unknown')}")
print(f"工作目录:      {os.getcwd()}")

系统:          Linux 4.19.90-vhulk2211.3.0.h1543.eulerosv2r10.aarch64
架构:          aarch64
主机名:        notebook-c768c7a7-f8ad-41b7-91cb-f28aa622b000
当前用户:      unknown
工作目录:      /home/ma-user/work


## 4. 磁盘与内存

In [5]:
print("--- 磁盘 ---")
for line in os.popen("df -h /home/ma-user/work").read().strip().split("\n"):
    print(line)

print("\n--- 内存 ---")
for line in os.popen("free -h").read().strip().split("\n"):
    print(line)

--- 磁盘 ---
Filesystem      Size  Used Avail Use% Mounted on
/dev/sdb        196G  116K  196G   1% /home/ma-user/work

--- 内存 ---
total        used        free      shared  buff/cache   available
Mem:           1.5Ti        25Gi       1.4Ti       120Mi       3.5Gi       1.4Ti
Swap:             0B          0B          0B


## 5. CANN / NPU 驱动

In [6]:
print("--- npu-smi info ---")
for line in os.popen("npu-smi info 2>/dev/null || echo 'npu-smi 不可用'").read().strip().split("\n"):
    print(line)

print("\n--- CANN 版本文件 ---")
cann_version_file = "/usr/local/Ascend/ascend-toolkit/latest/version.cfg"
if os.path.exists(cann_version_file):
    for line in open(cann_version_file).readlines():
        print(f"  {line.rstrip()}")
else:
    # 尝试其他路径
    for line in os.popen("find /usr/local/Ascend -name 'version.cfg' 2>/dev/null | head -3").read().strip().split("\n"):
        if line:
            print(f"  找到: {line}")
            for l in open(line).readlines():
                print(f"    {l.rstrip()}")

--- npu-smi info ---
+------------------------------------------------------------------------------------------------+
| npu-smi 23.0.6                   Version: 23.0.6                                               |
+---------------------------+---------------+----------------------------------------------------+
| NPU   Name                | Health        | Power(W)    Temp(C)           Hugepages-Usage(page)|
| Chip                      | Bus-Id        | AICore(%)   Memory-Usage(MB)  HBM-Usage(MB)        |
+===========================+===============+====================================================+
| 0     910B4               | OK            | 88.8        40                0    / 0             |
| 0                         | 0000:C1:00.0  | 0           0    / 0          2887 / 32768         |
+===========================+===============+====================================================+
+---------------------------+---------------+-------------------------------------------

## 6. pip 包列表

In [7]:
import subprocess
result = subprocess.run(["pip", "list"], capture_output=True, text=True)
lines = result.stdout.strip().split("\n")
print(f"共 {len(lines)-2} 个包，前 50 个:")
for line in lines[:52]:
    print(line)

共 273 个包，前 50 个:
Package                                  Version
---------------------------------------- ------------
absl-py                                  2.4.0
accelerate                               1.0.1
addict                                   2.4.0
aiohappyeyeballs                         2.6.1
aiohttp                                  3.13.5
aiosignal                                1.4.0
albumentations                           1.3.1
antlr4-python3-runtime                   4.9.3
arrow                                    1.4.0
asc_op_compile_base                      0.1.0
asc_opc_tool                             0.1.0
astroid                                  3.3.11
asttokens                                3.0.1
attrs                                    23.2.0
audioread                                3.1.0
auto_tune                                0.1.0
binaryornot                              0.4.4
blinker                                  1.9.0
blobfile                       